# DS-ST Paper Analysis & Ablation Study

**Datasets required (already on Kaggle):**
1. `mashwoo/set-transformer-10mb` — model code
2. `nguyenmanhhust/noc-10mb` — pretrained checkpoint
3. `nguyenmanhhust/gf-kit-file` — GF dataset (gf_groups.csv)

**GPU**: T4 x2 or P100

**Outputs:**
- Confusion matrix (aggregate + per-fold)
- Per-class precision / recall / F1 (mean +/- std)
- Mixture-only metrics (NOC >= 2)
- DS-ST vs deepNoC comparison table
- Ablation study (7 variants x 5 folds)
- All LaTeX-ready tables

In [ ]:
import sys, os, warnings, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    precision_score, recall_score,
)
from scipy import stats
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter

warnings.filterwarnings('ignore')

sys.path.insert(0, '/kaggle/input/datasets/mashwoo/set-transformer-10mb')
DATA_CSV  = '/kaggle/input/datasets/nguyenmanhhust/gf-kit-file/gf_groups.csv'
CKPT_PATH = '/kaggle/input/datasets/nguyenmanhhust/noc-10mb/noc_10mb'

from set_transformer_10mb import SetTransformerMixture

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_NOC = 5
NOC_NAMES = [f'NOC={k}' for k in range(1, N_NOC + 1)]
print(f'Device: {DEVICE}')

## 1. Data Loading & Preprocessing

In [ ]:
df = pd.read_csv(DATA_CSV)
print(f'Dataset: {df.shape}')

SKIP = {'TPH', 'PeakCount', 'MaxHeight'}
ALLELE_MAP = {'X': -2.0, 'Y': -1.0}
LOCUS_COLS = [c for c in df.columns
              if c not in ('target_noc', 'group_id')
              and c.rsplit('_', 1)[1] not in SKIP]

loci_names = sorted(set(c.rsplit('_', 1)[0] for c in LOCUS_COLS))
locus2idx  = {l: i for i, l in enumerate(loci_names)}

col_info = []
for col in LOCUS_COLS:
    ln, als = col.rsplit('_', 1)
    li = locus2idx.get(ln, 0)
    av = ALLELE_MAP.get(als, None)
    if av is None:
        try: av = float(als)
        except: av = 0.0
    col_info.append((li, av))

noc       = df['target_noc'].values
group_ids = df['group_id'].values
vals      = df[LOCUS_COLS].values.astype(np.float32)
labels    = noc - 1
print(f'NOC dist: { {k: int((noc==k).sum()) for k in sorted(set(noc))} }')

S = 200
tokens_3d = np.zeros((len(df), S, 3), dtype=np.float32)
masks     = np.zeros((len(df), S),    dtype=np.float32)
for i in range(len(df)):
    pos = 0
    for j, (li, av) in enumerate(col_info):
        h = vals[i, j]
        if h <= 0 or np.isnan(h): continue
        if pos < S:
            tokens_3d[i, pos, 0] = li
            tokens_3d[i, pos, 1] = av
            tokens_3d[i, pos, 2] = np.log1p(h)
            masks[i, pos] = 1.0
            pos += 1

max_len   = int(masks.sum(1).max()) + 2
tokens_3d = tokens_3d[:, :max_len, :]
masks     = masks[:, :max_len]
print(f'Token shape: {tokens_3d.shape}, max_len: {max_len}')

In [ ]:
def enrich(tokens, masks):
    N, S, _ = tokens.shape
    e = np.zeros((N, S, 8), dtype=np.float32)
    e[:, :, :3] = tokens
    for i in range(N):
        vi = np.where(masks[i] > 0)[0]
        if len(vi) == 0: continue
        loci  = tokens[i, vi, 0].astype(int)
        raw_h = np.expm1(tokens[i, vi, 2])
        h_max = raw_h.max() + 1e-9
        lp = {}
        for j, idx in enumerate(vi):
            lp.setdefault(int(loci[j]), []).append((idx, tokens[i, idx, 1], raw_h[j]))
        for l, peaks in lp.items():
            n_l  = len(peaks); lsum = sum(p[2] for p in peaks) + 1e-9
            ps   = sorted(peaks, key=lambda x: -x[2])
            ah   = {round(p[1], 1): p[2] for p in peaks}
            for rank, (idx, av, h) in enumerate(ps):
                ph = ah.get(round(av + 1.0, 1), 0.0)
                e[i, idx, 3] = h / lsum
                e[i, idx, 4] = h / (ph + 1e-9) if ph > 0 else 0.0
                e[i, idx, 5] = rank / max(n_l - 1, 1)
                e[i, idx, 6] = n_l / 12.0
                e[i, idx, 7] = h / h_max
    return e

tokens_8d = enrich(tokens_3d, masks)
tokens_8d_zeros = np.concatenate(
    [tokens_3d, np.zeros((*tokens_3d.shape[:2], 5), dtype=np.float32)], axis=-1)
print(f'Enriched shape: {tokens_8d.shape}')

## 2. Model Definitions

In [ ]:
import io, zipfile

def _load_ckpt(path):
    try:
        with zipfile.ZipFile(path, 'r') as zf:
            buf = io.BytesIO()
            with zipfile.ZipFile(buf, 'w') as out:
                for name in zf.namelist():
                    out.writestr(name, zf.read(name))
            buf.seek(0)
            return torch.load(buf, map_location='cpu', weights_only=False)
    except zipfile.BadZipFile:
        return torch.load(path, map_location='cpu', weights_only=False)


def load_backbone(pretrain=True, arch_overrides=None):
    sd = _load_ckpt(CKPT_PATH)
    owner_lut = sd.get('owner_lut', None)
    kwargs = dict(
        n_loci=24, d_locus=16, d_model=128, n_heads=4, n_isab=2, m_inducing=32,
        n_classes=45, n_noc=5, dropout=0.1, cls_decoder='pooled',
        n_token_feats=8, encoder='isab++', num_embed='periodic',
        n_freq=8, d_num_emb=8, periodic_sigma=0.3, nc_attn='mab0',
        feas_filter=True, set_of_set=True, owner_lut=owner_lut,
        aux_heads=True, noc_head_v2=True,
    )
    if arch_overrides:
        kwargs.update(arch_overrides)
        print(f'  Arch overrides: {arch_overrides}')
    model = SetTransformerMixture(**kwargs)
    if pretrain:
        missing, unexpected = model.load_state_dict(sd, strict=False)
        n_ok = len(sd) - len(unexpected)
        print(f'  Pretrain: loaded {n_ok}/{len(sd)} tensors')
    else:
        print('  Random init')
    return model


class NOCFinetune(nn.Module):
    def __init__(self, backbone, d_model=128, n_noc=5, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.noc_head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(d_model, 64),
            nn.ReLU(True), nn.Dropout(dropout), nn.Linear(64, n_noc),
        )

    def forward(self, tokens, mask):
        return self.noc_head(self.backbone.encode(tokens, mask.bool()))


class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma; self.weight = weight

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()


def oversample_minority(indices, labels):
    counts = Counter(labels[indices].tolist())
    target = max(counts.values())
    out = []
    for cls, cnt in counts.items():
        idx = indices[labels[indices] == cls]
        if cnt < target:
            reps = target // cnt; extra = target % cnt
            idx  = np.concatenate([np.tile(idx, reps),
                                   np.random.choice(idx, extra, replace=False)])
        out.append(idx)
    return np.concatenate(out)


def train_fold(model, tok_tr, msk_tr, lbl_tr, tok_val, msk_val, lbl_val,
               device, loss_kind='focal', epochs=60, lr=3e-4, bs=64,
               patience=12, do_oversample=True):
    unique, counts = np.unique(lbl_tr, return_counts=True)
    total = counts.sum()
    cw = torch.tensor([total / (len(unique) * c) for c in counts],
                      dtype=torch.float32).to(device)
    criterion = FocalLoss(weight=cw) if loss_kind == 'focal' else nn.CrossEntropyLoss(weight=cw)

    if do_oversample:
        tr_idx = oversample_minority(np.arange(len(lbl_tr)), lbl_tr)
    else:
        tr_idx = np.arange(len(lbl_tr))
    np.random.shuffle(tr_idx)
    dl = DataLoader(TensorDataset(
        torch.tensor(tok_tr[tr_idx], dtype=torch.float32),
        torch.tensor(msk_tr[tr_idx], dtype=torch.float32),
        torch.tensor(lbl_tr[tr_idx], dtype=torch.long)),
        batch_size=bs, shuffle=True, drop_last=True)
    Xv = torch.tensor(tok_val, dtype=torch.float32).to(device)
    Mv = torch.tensor(msk_val, dtype=torch.float32).to(device)
    Yv = torch.tensor(lbl_val, dtype=torch.long)

    for p in model.backbone.parameters(): p.requires_grad = False
    opt = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr * 3, weight_decay=1e-4)
    model.train()
    for _ in range(5):
        for xb, mb, yb in dl:
            xb, mb, yb = xb.to(device), mb.to(device), yb.to(device)
            opt.zero_grad(); criterion(model(xb, mb), yb).backward(); opt.step()
    for p in model.backbone.parameters(): p.requires_grad = True

    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    best_macro, best_state, wait = 0.0, None, 0

    for ep in range(epochs):
        model.train()
        for xb, mb, yb in dl:
            xb, mb, yb = xb.to(device), mb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb, mb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            preds = torch.cat([model(Xv[i:i+bs], Mv[i:i+bs]).argmax(1)
                               for i in range(0, len(Xv), bs)]).cpu().numpy()
        macro = f1_score(Yv.numpy(), preds, average='macro')
        if macro > best_macro:
            best_macro = macro
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if (ep + 1) % 10 == 0 or wait == 0:
            print(f'    Ep {ep+1:3d}: val_macro={macro:.4f}  best={best_macro:.4f}')
        if wait >= patience:
            print(f'    Early stop ep {ep+1}'); break

    model.load_state_dict(best_state)
    return best_macro

print('Model definitions ready.')

## 3. Paper Analysis — Full DS-ST (5-fold)

Generates: confusion matrix, per-class metrics, mixture-only metrics, deepNoC comparison.

In [ ]:
# deepNoC Table 2 exact numbers (Taylor & Humphries 2024)
DEEPNOC = {
    'precision': [1.000, 1.000, 0.873, 0.882, 0.782],
    'recall':    [1.000, 0.967, 0.958, 0.788, 0.859],
    'f1':        [1.000, 0.983, 0.914, 0.832, 0.819],
    'macro_f1':  0.910,
}

gkf = GroupKFold(n_splits=5)
all_true, all_pred = [], []
fold_results = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(tokens_8d, labels, groups=group_ids), 1):
    print(f'\n{"="*50}  Fold {fold}  {"="*50}')
    t0    = time.time()
    model = NOCFinetune(load_backbone()).to(DEVICE)
    macro = train_fold(model,
                       tokens_8d[tr_idx], masks[tr_idx], labels[tr_idx],
                       tokens_8d[val_idx], masks[val_idx], labels[val_idx],
                       device=DEVICE)

    model.eval()
    Xv = torch.tensor(tokens_8d[val_idx], dtype=torch.float32).to(DEVICE)
    Mv = torch.tensor(masks[val_idx],     dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        preds = torch.cat([model(Xv[i:i+64], Mv[i:i+64]).argmax(1)
                           for i in range(0, len(Xv), 64)]).cpu().numpy()

    y_true = labels[val_idx]
    all_true.append(y_true)
    all_pred.append(preds)

    per_class_f1 = f1_score(y_true, preds, average=None, labels=list(range(N_NOC)))
    per_class_p  = precision_score(y_true, preds, average=None, labels=list(range(N_NOC)))
    per_class_r  = recall_score(y_true, preds, average=None, labels=list(range(N_NOC)))

    mix_mask = y_true >= 1
    macro_mix = f1_score(y_true[mix_mask], preds[mix_mask], average='macro',
                         labels=[1, 2, 3, 4]) if mix_mask.sum() > 0 else 0.0

    fold_results.append({
        'fold': fold,
        'macro_f1': macro,
        'micro_f1': f1_score(y_true, preds, average='micro'),
        'macro_f1_mix': macro_mix,
        'per_class_f1': per_class_f1.tolist(),
        'per_class_precision': per_class_p.tolist(),
        'per_class_recall': per_class_r.tolist(),
        'cm': confusion_matrix(y_true, preds, labels=list(range(N_NOC))).tolist(),
        'n_samples': len(y_true),
        'elapsed_s': time.time() - t0,
    })

    print(f'\n  Fold {fold}: Macro={macro:.4f}  Macro(mix)={macro_mix:.4f}  ({time.time()-t0:.0f}s)')
    print(classification_report(y_true, preds, target_names=NOC_NAMES, digits=3))

In [ ]:
print('=' * 70)
print('  PAPER ANALYSIS RESULTS')
print('=' * 70)

macro_f1s = [r['macro_f1'] for r in fold_results]
print(f'\n[1] DS-ST Macro-F1: {np.mean(macro_f1s):.4f} +/- {np.std(macro_f1s):.4f}')
print(f'    Per-fold: {["%.4f" % m for m in macro_f1s]}')

macro_mix = [r['macro_f1_mix'] for r in fold_results]
print(f'\n[2] DS-ST Macro-F1 (mixtures only, NOC>=2): {np.mean(macro_mix):.4f} +/- {np.std(macro_mix):.4f}')

print(f'\n[3] Per-Class Metrics (mean +/- std across 5 folds):')
print(f'    {"Class":<8} {"Precision":>12} {"Recall":>12} {"F1":>12}')
print(f'    {"-"*8} {"-"*12} {"-"*12} {"-"*12}')
for c in range(N_NOC):
    ps = [r['per_class_precision'][c] for r in fold_results]
    rs = [r['per_class_recall'][c] for r in fold_results]
    fs = [r['per_class_f1'][c] for r in fold_results]
    print(f'    NOC={c+1:<3} {np.mean(ps):.3f}+/-{np.std(ps):.3f}  '
          f'{np.mean(rs):.3f}+/-{np.std(rs):.3f}  '
          f'{np.mean(fs):.3f}+/-{np.std(fs):.3f}')

all_t = np.concatenate(all_true)
all_p = np.concatenate(all_pred)
cm_agg = confusion_matrix(all_t, all_p, labels=list(range(N_NOC)))
print(f'\n[4] Aggregate Confusion Matrix (all folds, n={len(all_t)}):')
print(f'    Pred->  {"  ".join(NOC_NAMES)}')
for i in range(N_NOC):
    row = '  '.join(f'{cm_agg[i,j]:5d}' for j in range(N_NOC))
    print(f'    {NOC_NAMES[i]}  {row}')

print(f'\n[5] DS-ST vs deepNoC - Per-Class Recall:')
print(f'    {"Class":<8} {"DS-ST":>8} {"deepNoC":>8} {"Delta":>8}')
print(f'    {"-"*8} {"-"*8} {"-"*8} {"-"*8}')
dsst_recall = [np.mean([r['per_class_recall'][c] for r in fold_results]) for c in range(N_NOC)]
for c in range(N_NOC):
    d = dsst_recall[c] - DEEPNOC['recall'][c]
    print(f'    NOC={c+1:<3} {dsst_recall[c]:>8.3f} {DEEPNOC["recall"][c]:>8.3f} {d:>+8.3f}')
print(f'    {"Macro F1":<8} {np.mean(macro_f1s):>8.3f} {DEEPNOC["macro_f1"]:>8.3f} '
      f'{np.mean(macro_f1s) - DEEPNOC["macro_f1"]:>+8.3f}')

### LaTeX Tables

In [ ]:
print('=== CONFUSION MATRIX (LaTeX) ===')
print(r'\begin{table}[htbp]')
print(r'  \centering')
print(r'  \caption{Aggregate NOC Confusion Matrix (5-Fold GroupKFold, $n = ' + str(len(all_t)) + r'$).}')
print(r'  \label{tab:confusion}')
print(r'  \begin{tabular}{l' + 'c' * N_NOC + '}')
print(r'    \toprule')
print(r'    & \multicolumn{' + str(N_NOC) + r'}{c}{\textbf{Predicted NOC}} \\')
print(r'    \cmidrule(lr){2-' + str(N_NOC + 1) + '}')
hdr = ' & '.join([r'\textbf{' + str(k) + '}' for k in range(1, N_NOC + 1)])
print(r'    \textbf{True NOC} & ' + hdr + r' \\')
print(r'    \midrule')
for i in range(N_NOC):
    cells = []
    for j in range(N_NOC):
        v = cm_agg[i, j]
        cells.append(r'\textbf{' + str(v) + '}' if i == j else str(v))
    print(f'    {i+1} & ' + ' & '.join(cells) + r' \\')
print(r'    \bottomrule')
print(r'  \end{tabular}')
print(r'\end{table}')

print()
print('=== PER-CLASS METRICS (LaTeX) ===')
print(r'\begin{table}[htbp]')
print(r'  \centering')
print(r'  \caption{DS-ST Per-Class Metrics (mean $\pm$ std, 5-Fold GroupKFold).}')
print(r'  \label{tab:perclass_full}')
print(r'  \begin{tabular}{lccc}')
print(r'    \toprule')
print(r'    \textbf{NOC} & \textbf{Precision} & \textbf{Recall} & \textbf{$F_1$} \\')
print(r'    \midrule')
for c in range(N_NOC):
    ps = [r['per_class_precision'][c] for r in fold_results]
    rs = [r['per_class_recall'][c] for r in fold_results]
    fs = [r['per_class_f1'][c] for r in fold_results]
    print(f'    {c+1} & ${np.mean(ps):.3f} \\pm {np.std(ps):.3f}$'
          f' & ${np.mean(rs):.3f} \\pm {np.std(rs):.3f}$'
          f' & ${np.mean(fs):.3f} \\pm {np.std(fs):.3f}$ \\\\')
print(r'    \bottomrule')
print(r'  \end{tabular}')
print(r'\end{table}')

In [ ]:
# Save full results to JSON
output = {
    'fold_results': fold_results,
    'aggregate_cm': cm_agg.tolist(),
    'dsst_macro_f1': float(np.mean(macro_f1s)),
    'dsst_macro_f1_std': float(np.std(macro_f1s)),
    'dsst_macro_f1_mix': float(np.mean(macro_mix)),
    'dsst_per_class_recall': dsst_recall,
    'deepnoc': DEEPNOC,
}
with open('paper_analysis_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print('Saved paper_analysis_results.json')

## 4. Ablation Study (7 variants x 5 folds)

| Variant | What it removes |
|---|---|
| no_pretrain | No 10MB pre-training, random init |
| no_enrich | No peak enrichment features (3d tokens instead of 8d) |
| ce_loss | Cross-entropy instead of Focal loss |
| no_oversample | No minority oversampling |
| softmax_attn | Standard softmax instead of sigmoid gate (mab0) |
| no_set_of_set | Disable set-of-set locus partitioning |
| linear_embed | Linear embedding instead of PeriodicPLR |

In [ ]:
VARIANTS = {
    # Training ablations
    'no_pretrain':    {'pretrain': False, 'enrich': True,  'loss': 'focal', 'arch': {}},
    'no_enrich':      {'pretrain': True,  'enrich': False, 'loss': 'focal', 'arch': {}},
    'ce_loss':        {'pretrain': True,  'enrich': True,  'loss': 'ce',    'arch': {}},
    'no_oversample':  {'pretrain': True,  'enrich': True,  'loss': 'focal', 'arch': {}, 'oversample': False},
    # Architecture ablations
    'softmax_attn':   {'pretrain': False, 'enrich': True,  'loss': 'focal',
                       'arch': {'nc_attn': 'softmax'}},
    'no_set_of_set':  {'pretrain': False, 'enrich': True,  'loss': 'focal',
                       'arch': {'set_of_set': False}},
    'linear_embed':   {'pretrain': False, 'enrich': True,  'loss': 'focal',
                       'arch': {'num_embed': 'linear'}},
}

abl_results = {}

for var_name, cfg in VARIANTS.items():
    print(f'\n{"="*60}\n  VARIANT: {var_name}\n{"="*60}')
    tokens_use   = tokens_8d if cfg['enrich'] else tokens_8d_zeros
    do_os        = cfg.get('oversample', True)
    fold_macros  = []

    for fold, (tr_i, val_i) in enumerate(gkf.split(tokens_use, labels, groups=group_ids), 1):
        print(f'\n  Fold {fold}:')
        t0       = time.time()
        backbone = load_backbone(pretrain=cfg['pretrain'],
                                 arch_overrides=cfg.get('arch') or None)
        model    = NOCFinetune(backbone).to(DEVICE)

        macro = train_fold(model,
                           tokens_use[tr_i], masks[tr_i], labels[tr_i],
                           tokens_use[val_i], masks[val_i], labels[val_i],
                           device=DEVICE, loss_kind=cfg['loss'],
                           epochs=50, patience=10, do_oversample=do_os)

        model.eval()
        Xv = torch.tensor(tokens_use[val_i], dtype=torch.float32).to(DEVICE)
        Mv = torch.tensor(masks[val_i],      dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            preds = torch.cat([model(Xv[i:i+64], Mv[i:i+64]).argmax(1)
                               for i in range(0, len(Xv), 64)]).cpu().numpy()
        print(f'  Fold {fold}: Macro={macro:.4f}  ({time.time()-t0:.0f}s)')
        print(classification_report(labels[val_i], preds,
              target_names=NOC_NAMES, digits=3))
        fold_macros.append(macro)

    abl_results[var_name] = fold_macros
    print(f'\n  {var_name}: {np.mean(fold_macros):.4f} +/- {np.std(fold_macros):.4f}')

In [ ]:
FULL_MEAN, FULL_STD = np.mean(macro_f1s), np.std(macro_f1s)

print(f'\n{"="*65}\n  ABLATION SUMMARY\n{"="*65}')
print(f'  {"Variant":<20} {"Mean":>8}  {"Std":>6}  {"Delta":>8}')
print(f'  {"-"*20}  {"-"*6}  {"-"*5}  {"-"*7}')
print(f'  {"DS-ST (full)":<20} {FULL_MEAN:>8.4f}  {FULL_STD:>6.4f}  {"---":>8}')
for var_name, fms in abl_results.items():
    m = np.mean(fms)
    print(f'  {var_name:<20} {m:>8.4f}  {np.std(fms):>6.4f}  {m - FULL_MEAN:>+8.4f}')

# Paired t-test: DS-ST full vs each ablation
print(f'\n  Statistical significance (paired t-test vs DS-ST full):')
for var_name, fms in abl_results.items():
    t_stat, p_val = stats.ttest_rel(macro_f1s, fms)
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    print(f'    vs {var_name:<18}: t={t_stat:+.3f}, p={p_val:.4f} {sig}')

# LaTeX
LABELS = {
    'no_pretrain':   r'w/o pre-training',
    'no_enrich':     r'w/o peak enrichment',
    'ce_loss':       r'CE loss (no Focal)',
    'no_oversample': r'w/o oversampling',
    'softmax_attn':  r'Softmax attn (no $\sigma$-gate)',
    'no_set_of_set': r'w/o set-of-set',
    'linear_embed':  r'Linear embed (no Periodic)',
}

print(f'\n=== ABLATION TABLE (LaTeX) ===')
print(r'\begin{table}[htbp]')
print(r'  \centering')
print(r'  \caption{Ablation Study: Macro F1 on 5-Fold GroupKFold.}')
print(r'  \label{tab:ablation}')
print(r'  \begin{tabular}{lccc}')
print(r'    \toprule')
print(r'    \textbf{Configuration} & \textbf{Macro F1} & \textbf{$\Delta$} & \textbf{$p$} \\')
print(r'    \midrule')
print(f'    \\textbf{{DS-ST (full)}} & $\\mathbf{{{FULL_MEAN:.4f} \\pm {FULL_STD:.4f}}}$ & --- & --- \\\\')
print(r'    \midrule')
for var_name, fms in abl_results.items():
    m = np.mean(fms)
    s = np.std(fms)
    d = m - FULL_MEAN
    _, p_val = stats.ttest_rel(macro_f1s, fms)
    p_str = f'{p_val:.3f}' if p_val >= 0.001 else '<0.001'
    label = LABELS.get(var_name, var_name)
    print(f'    {label} & ${m:.4f} \\pm {s:.4f}$ & ${d:+.4f}$ & {p_str} \\\\')
print(r'    \bottomrule')
print(r'  \end{tabular}')
print(r'\end{table}')

In [ ]:
# Save ablation results
abl_output = {
    'full_system': {'macro_f1_per_fold': macro_f1s,
                    'mean': float(FULL_MEAN), 'std': float(FULL_STD)},
    'ablations': {k: {'per_fold': v,
                      'mean': float(np.mean(v)),
                      'std': float(np.std(v)),
                      'delta': float(np.mean(v) - FULL_MEAN),
                      'p_value': float(stats.ttest_rel(macro_f1s, v).pvalue)}
                 for k, v in abl_results.items()},
}
with open('ablation_results.json', 'w') as f:
    json.dump(abl_output, f, indent=2)
print('Saved ablation_results.json')
print('\nDone! Download paper_analysis_results.json and ablation_results.json.')

## 5. Paper Figures

Generate publication-quality figures for the paper:
1. Confusion matrix heatmap (aggregate across 5 folds)
2. Per-class F1 comparison: DS-ST vs deepNoC
3. Ablation delta bar chart

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Style ──
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'figure.dpi': 300,
})

# ── Figure 2: Confusion Matrix Heatmap ──
fig, ax = plt.subplots(figsize=(5.5, 4.5))

# Normalise rows to percentages
cm_pct = cm_agg.astype(float) / cm_agg.sum(axis=1, keepdims=True) * 100

# Annotation: count + percentage
annot = np.empty_like(cm_agg, dtype=object)
for i in range(N_NOC):
    for j in range(N_NOC):
        annot[i, j] = f'{cm_agg[i,j]}\n({cm_pct[i,j]:.1f}%)'

sns.heatmap(cm_pct, annot=annot, fmt='', cmap='Blues',
            xticklabels=[str(k) for k in range(1, N_NOC+1)],
            yticklabels=[str(k) for k in range(1, N_NOC+1)],
            vmin=0, vmax=100, cbar_kws={'label': 'Row %'},
            linewidths=0.5, linecolor='white', ax=ax)

ax.set_xlabel('Predicted NOC')
ax.set_ylabel('True NOC')
ax.set_title(f'DS-ST Confusion Matrix (n = {len(all_t):,})')
plt.tight_layout()
fig.savefig('fig_confusion_matrix.pdf', bbox_inches='tight')
fig.savefig('fig_confusion_matrix.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved fig_confusion_matrix.pdf / .png')

In [ ]:
# ── Figure 3: Per-Class F1 — DS-ST vs deepNoC ──
fig, ax = plt.subplots(figsize=(6, 4))

dsst_f1_mean = [np.mean([r['per_class_f1'][c] for r in fold_results]) for c in range(N_NOC)]
dsst_f1_std  = [np.std([r['per_class_f1'][c] for r in fold_results]) for c in range(N_NOC)]
deepnoc_f1   = DEEPNOC['f1']

x = np.arange(N_NOC)
w = 0.35

bars1 = ax.bar(x - w/2, dsst_f1_mean, w, yerr=dsst_f1_std,
               label='DS-ST', color='#2196F3', edgecolor='white',
               capsize=3, error_kw={'linewidth': 1})
bars2 = ax.bar(x + w/2, deepnoc_f1, w,
               label='deepNoC', color='#FF9800', edgecolor='white')

# Value labels
for bar, val in zip(bars1, dsst_f1_mean):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)
for bar, val in zip(bars2, deepnoc_f1):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('NOC')
ax.set_ylabel('$F_1$ Score')
ax.set_title('Per-Class $F_1$: DS-ST vs deepNoC')
ax.set_xticks(x)
ax.set_xticklabels([str(k) for k in range(1, N_NOC+1)])
ax.set_ylim(0.75, 1.05)
ax.legend(loc='lower left', framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig('fig_perclass_f1_comparison.pdf', bbox_inches='tight')
fig.savefig('fig_perclass_f1_comparison.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved fig_perclass_f1_comparison.pdf / .png')

In [ ]:
# ── Figure 4: Ablation Impact ──
fig, ax = plt.subplots(figsize=(7, 4))

abl_names = list(abl_results.keys())
abl_labels = {
    'no_pretrain':   'w/o pre-training',
    'no_enrich':     'w/o peak enrichment',
    'ce_loss':       'CE loss (no Focal)',
    'no_oversample': 'w/o oversampling',
    'softmax_attn':  'Softmax attn',
    'no_set_of_set': 'w/o set-of-set',
    'linear_embed':  'Linear embed',
}
labels = [abl_labels.get(n, n) for n in abl_names]
deltas = [np.mean(abl_results[n]) - FULL_MEAN for n in abl_names]
stds   = [np.std(abl_results[n]) for n in abl_names]
pvals  = [stats.ttest_rel(macro_f1s, abl_results[n]).pvalue for n in abl_names]

colors = ['#4CAF50' if d > 0 else '#F44336' if p < 0.05 else '#FF9800'
          for d, p in zip(deltas, pvals)]

y_pos = np.arange(len(labels))
ax.barh(y_pos, [d * 100 for d in deltas], color=colors, edgecolor='white',
        height=0.6)

for i, (d, p) in enumerate(zip(deltas, pvals)):
    sig = '**' if p < 0.01 else '*' if p < 0.05 else ''
    ax.text(d * 100 + (0.15 if d >= 0 else -0.15),
            i, f'{d*100:+.2f} pp{sig}',
            va='center', ha='left' if d >= 0 else 'right', fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlabel('$\\Delta$ Macro F1 (percentage points)')
ax.set_title('Ablation Study: Impact of Each Component')
ax.axvline(x=0, color='gray', linewidth=0.8, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.invert_yaxis()

plt.tight_layout()
fig.savefig('fig_ablation_delta.pdf', bbox_inches='tight')
fig.savefig('fig_ablation_delta.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved fig_ablation_delta.pdf / .png')

In [ ]:
# ── List all output files ──
import glob
outputs = sorted(glob.glob('*.json') + glob.glob('*.pdf') + glob.glob('*.png'))
print('\n' + '='*50)
print('  OUTPUT FILES — download all of these')
print('='*50)
for f in outputs:
    sz = os.path.getsize(f)
    print(f'  {f:<40} {sz:>10,} bytes')
print(f'\nTotal: {len(outputs)} files')